In [ ]:
# Install first:
# pip install transformers torch accelerate

*********************
import torch

# You can replace this with another Hugging Face model ID
*********************

# 1. Load tokenizer from Hugging Face
*********************
# AutoTokenizer automatically loads the tokenizer that belongs to the model.

# 2. Load model architecture + weights from Hugging Face
*********************
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# 3. Create prompt
prompt = "Explain why accessing model weights directly can be better than using Ollama."

messages = [
    {"role": "system", "content": "You are a clear and concise AI assistant."},
*********************
]

# 4. Format prompt for instruction-tuned model
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 5. Convert text to model inputs
inputs = tokenizer(text, return_tensors="pt").to(model.device) # pt means PyTorch

# 6. Generate response locally
with torch.no_grad():
*********************
        **inputs, # **inputs means "unpack this dictionary into named arguments"
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True # do_sample=True means the model chooses the next token probabilistically, not always by the highest score.
    )

# 7. Decode response
response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)

In [ ]:
# Because the tokenizer may return several fields, such as:

# {
#     "input_ids": ...,
#     "attention_mask": ...,
#     "token_type_ids": ...
# }

# Using **inputs passes all needed fields automatically.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

*********************
LOCAL_DIR = "./qwen_local_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Save model files into your current Colab directory
tokenizer.save_pretrained(LOCAL_DIR)
model.save_pretrained(LOCAL_DIR)

print(f"Model saved to: {LOCAL_DIR}")

In [ ]:
# pip install openai

import os
from openai import OpenAI
*********************

hf_token = getpass("Enter your new Hugging Face token: ")

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token,
)

*********************
    model="katanemo/Arch-Router-1.5B:hf-inference",
    messages=[
        {
            "role": "user",
            "content": "Explain Hugging Face Inference Providers in simple terms."
        }
    ],
    max_tokens=200,
)

print(completion.choices[0].message.content)

In [ ]:
# api_key="hf_..."

In [ ]:
# !pip install transformers torch
from transformers import pipeline

model_name = "Falconsai/text_summarization"

*********************


text = """
Artificial intelligence is changing the way people work, study, and communicate.
In education, AI tools can help students summarize long articles, explain difficult
concepts, and generate practice questions. However, students should still learn how
to evaluate the quality of AI-generated answers because models can make mistakes.
Teachers can use AI as a support tool, but not as a replacement for critical thinking.
"""

summary = summarizer(
    text,
    max_length=50,
    min_length=20,
    do_sample=False
)

# print(summary[0]["summary_text"])
print(summary[0]["generated_text"])

In [ ]:

# # !pip install transformers torch
# from transformers import pipeline

# model_name = "Falconsai/text_summarization"

# summarizer = pipeline("summarization", model=model_name)
# # summarizer = pipeline("text-generation", model=model_name)

# text = """
# Artificial intelligence is changing the way people work, study, and communicate.
# In education, AI tools can help students summarize long articles, explain difficult
# concepts, and generate practice questions. However, students should still learn how
# to evaluate the quality of AI-generated answers because models can make mistakes.
# Teachers can use AI as a support tool, but not as a replacement for critical thinking.
# """

# summary = summarizer(
#     text,
#     max_length=50,
#     min_length=20,
#     do_sample=False
# )

# print(summary[0]["summary_text"])
# # print(summary[0]["generated_text"])

In [ ]:
# Install if needed
# !pip install -q transformers torch sentencepiece
#-q means quiet

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "Falconsai/text_summarization"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = """
Artificial intelligence is changing the way people work, study, and communicate.
In education, AI tools can help students summarize long articles, explain difficult
concepts, and generate practice questions. However, students should still learn how
to evaluate the quality of AI-generated answers because models can make mistakes.
Teachers can use AI as a support tool, but not as a replacement for critical thinking.
"""

input_text = "summarize: " + text

inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

with torch.no_grad():  # Disables gradient calculation because we are only using the model for inference
    summary_ids = model.generate(
        inputs["input_ids"],      # The tokenized input text
        max_length=60,            # Maximum length of the generated summary
        min_length=20,            # Minimum length of the generated summary
        num_beams=4,              # Uses beam search with 4 candidate summaries
        length_penalty=2.0,       # Penalizes very long summaries
        early_stopping=True       # Stops generation when a good complete summary is found
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

In [ ]:
summary

In [ ]:
# !pip install datasets

from datasets import load_dataset

*********************

print(dataset)
print(dataset["train"][0])

**Pretrained Model**

In [ ]:
!pip install -q transformers torch sentencepiece datasets evaluate rouge_score sacrebleu nltk bert_score

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm

dataset = load_dataset("knkarthick/samsum")

model_name = "Falconsai/text_summarization"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(dataset)
print("Device:", device)


In [ ]:
test_data = dataset["test"].select(range(100)) #using a small subset

In [ ]:
def generate_summary(text):
    input_text = "summarize: " + text

*********************
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
*********************
            inputs["input_ids"],
            max_length=80,
            min_length=10,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

*********************
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
predictions = []
references = []

for example in tqdm(test_data):
    dialogue = example["dialogue"]
    reference_summary = example["summary"]

    generated_summary = generate_summary(dialogue)

*********************
*********************

print("Example prediction:")
print(predictions[0])

print("\nReference summary:")
print(references[0])

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

*********************

bleu_results = bleu.compute(predictions=predictions, references=[[ref] for ref in references])

meteor_results = meteor.compute(predictions=predictions, references=references)

bertscore_results = bertscore.compute(predictions=predictions, references=references, lang="en")

results = {
    "rouge1": rouge_results["rouge1"],
    "rouge2": rouge_results["rouge2"],
    "rougeL": rouge_results["rougeL"],
    "rougeLsum": rouge_results["rougeLsum"],
*********************
*********************
    "bertscore_precision": np.mean(bertscore_results["precision"]),
    "bertscore_recall": np.mean(bertscore_results["recall"]),
    "bertscore_f1": np.mean(bertscore_results["f1"])
}

results

In [ ]:
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

**Finetuned Model**

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

dataset = load_dataset("knkarthick/samsum")

model_name = "Falconsai/text_summarization"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(dataset)
print("Device:", device)

In [ ]:
# Use small subsets
*********************
validation_data = dataset["validation"].select(range(100))
test_data = dataset["test"].select(range(100))

In [ ]:
# Preprocess the dataset
max_input_length = 512
max_target_length = 80

def preprocess_function(examples):
    inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]
    targets = examples["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_train = train_data.map(
    preprocess_function,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_validation = validation_data.map(
    preprocess_function,
    batched=True,
    remove_columns=validation_data.column_names
)

In [ ]:
*********************
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Creates a data collator that prepares batches during training
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,  # Uses the tokenizer to pad input_ids and labels correctly
    model=model           # Uses the model structure to prepare decoder inputs when needed
)

# Defines the training settings
training_args = Seq2SeqTrainingArguments(
    output_dir="./finetuned_samsum_model",  # Folder where checkpoints and model files will be saved
    eval_strategy="epoch",                  # Evaluate the model at the end of each epoch
    save_strategy="epoch",                  # Save a checkpoint at the end of each epoch
    learning_rate=2e-5,                     # Step size used by the optimizer during training
    per_device_train_batch_size=4,          # Number of training examples processed at once on each GPU or CPU
    per_device_eval_batch_size=4,           # Number of validation examples processed at once on each GPU or CPU
    weight_decay=0.01,                      # Regularization value to reduce overfitting
    save_total_limit=2,                     # Keep only the latest 2 saved checkpoints
    num_train_epochs=1,                     # Number of full passes through the training dataset
    predict_with_generate=True,             # Generate summaries during evaluation instead of only computing loss
    fp16=torch.cuda.is_available(),         # Use faster 16-bit training if a GPU is available
    logging_steps=50,                       # Print training logs every 50 steps
    report_to="none"                       # Disable reporting to external tools like Weights & Biases
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

In [ ]:
fine_tuned_model_path = "./finetuned_samsum_model_final"

trainer.save_model(fine_tuned_model_path)
tokenizer.save_pretrained(fine_tuned_model_path)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

fine_tuned_model_path = "./finetuned_samsum_model_final"

tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(fine_tuned_model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [ ]:
def generate_summary(text):
    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=80,
            min_length=10,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
from tqdm import tqdm

predictions_finetuned = []
references_finetuned = []

for example in tqdm(test_data):
    dialogue = example["dialogue"]
    reference_summary = example["summary"]

    generated_summary = generate_summary(dialogue)

    predictions_finetuned.append(generated_summary)
    references_finetuned.append(reference_summary)

print("Example fine-tuned prediction:")
print(predictions_finetuned[0])

print("\nReference summary:")
print(references_finetuned[0])

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

rouge_results_finetuned = rouge.compute(predictions=predictions_finetuned, references=references_finetuned)

bleu_results_finetuned = bleu.compute(predictions=predictions_finetuned, references=[[ref] for ref in references_finetuned])

meteor_results_finetuned = meteor.compute(predictions=predictions_finetuned, references=references_finetuned)

bertscore_results_finetuned = bertscore.compute(predictions=predictions_finetuned, references=references_finetuned, lang="en")

results_finetuned = {
    "rouge1": rouge_results_finetuned["rouge1"],
    "rouge2": rouge_results_finetuned["rouge2"],
    "rougeL": rouge_results_finetuned["rougeL"],
    "rougeLsum": rouge_results_finetuned["rougeLsum"],
    "bleu": bleu_results_finetuned["bleu"],
    "meteor": meteor_results_finetuned["meteor"],
    "bertscore_precision": np.mean(bertscore_results_finetuned["precision"]),
    "bertscore_recall": np.mean(bertscore_results_finetuned["recall"]),
    "bertscore_f1": np.mean(bertscore_results_finetuned["f1"])
}

results_finetuned

In [ ]:
for metric, value in results_finetuned.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
print("Metric Comparison")
print("-" * 60)

for metric in results.keys():
    pretrained_score = results[metric]
    finetuned_score = results_finetuned[metric]
    difference = finetuned_score - pretrained_score

    print(
        f"{metric}: "
        f"pretrained = {pretrained_score:.4f}, "
        f"fine-tuned = {finetuned_score:.4f}, "
        f"difference = {difference:.4f}"
    )

**User Interface**

In [ ]:
!pip install -q gradio

*********************

def summarize_text(text):
    summary = generate_summary(text)
    return summary

*********************
*********************
    inputs=gr.Textbox(
        lines=8,
        placeholder="Enter a dialogue or paragraph here..."
    ),
    outputs=gr.Textbox(label="Generated Summary"),
    title="Text Summarization Demo",
    description="This app summarizes input text using a fine-tuned T5-based model."
)

*********************
